# Known-k temporal GCN: защищённый pilot
Отдельный эксперимент с двумя наблюдениями и заранее известным числом источников. Старые модели и отчёты не изменяются. Выполняй ячейки сверху вниз; после smoke остановись и пришли вывод. Никакие test или independent holdout здесь не открываются.

In [ ]:
from pathlib import Path
import importlib.util
import json
import subprocess
import sys

if importlib.util.find_spec('google.colab') is not None:
    if any(name == 'diffusion_sources' or name.startswith('diffusion_sources.') for name in sys.modules):
        raise RuntimeError('Код проекта уже импортирован: перезапусти среду Colab перед повторным setup')
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/diffusion-sources-known-k')
    REMOTE = 'https://github.com/1habibi/diffusion-sources-localization-.git'
    BRANCH = 'experiment/known-k-temporal-gcn'
    if not PROJECT.exists():
        subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REMOTE, str(PROJECT)], check=True)
    else:
        if not (PROJECT / '.git').exists():
            raise RuntimeError(f'{PROJECT} существует, но это не Git-клон; не перезаписываем')
        clean = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True).strip()
        branch = subprocess.check_output(['git', '-C', str(PROJECT), 'branch', '--show-current'], text=True).strip()
        if clean or branch != BRANCH:
            raise RuntimeError('Клон изменён или открыта другая ветка; не перезаписываем')
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
    packages = {'numpy': 'numpy', 'networkx': 'networkx', 'scipy': 'scipy', 'sklearn': 'scikit-learn', 'yaml': 'PyYAML', 'tqdm': 'tqdm', 'torch_geometric': 'torch-geometric'}
    missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT), '--no-deps'], check=True)
else:
    PROJECT = next((parent for base in [Path.cwd(), *Path.cwd().parents] for parent in [base] if (parent / 'src/diffusion_sources/known_k_temporal_pilot.py').is_file()), None)
    if PROJECT is None:
        raise RuntimeError('Локальный проект не найден; открой notebook из корня нужной ветки')

sys.path.insert(0, str(PROJECT / 'src'))
import torch
from diffusion_sources import known_k_temporal_pilot as PILOT
from diffusion_sources.known_k_temporal_pilot import KnownKPaths
if not Path(PILOT.__file__).resolve().is_relative_to(PROJECT.resolve()):
    raise RuntimeError('Kernel импортировал другую копию проекта; перезапусти среду и setup')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
APPROVED_PILOT = False
revision = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Project:', PROJECT, 'revision:', revision)
print('Kernel:', sys.executable, 'module:', PILOT.__file__)
print('PyTorch:', torch.__version__, 'device:', DEVICE)
print('Setup не запускал freeze, smoke или обучение')


In [ ]:
# Проверь эти пути перед freeze: новый output не должен совпадать со старым.
if importlib.util.find_spec('google.colab') is None:
    raise RuntimeError('Эта ячейка предназначена для Colab с подключённым Drive')
ROOT = Path('/content/drive/MyDrive/diffusion-sources')
PATHS = KnownKPaths(
    data_dir=ROOT / 'data/facebook_main',
    frozen_s1b_dir=ROOT / 'reports/runs/facebook_main_v2/s1b/seed_7026',
    output_dir=ROOT / 'reports/runs/known_k_temporal_gcn_pilot/v1',
    config_path=PROJECT / 'configs/known_k_temporal_gcn_pilot.yaml',
)
for name in ('data_dir', 'frozen_s1b_dir', 'config_path'):
    path = getattr(PATHS, name)
    if not path.exists():
        raise FileNotFoundError(f'{name}: {path}')
print('Data:', PATHS.data_dir)
print('Frozen S1b:', PATHS.frozen_s1b_dir)
print('New output:', PATHS.output_dir)
print('Config:', PATHS.config_path)
print('Пути проверены. Следующая ячейка — freeze; обучения ещё нет.')


In [ ]:
FROZEN = PILOT.run_stage('freeze', PATHS, torch.device('cpu'))
print(json.dumps(FROZEN, ensure_ascii=False, indent=2))
print('Freeze готов. Следующая ячейка — короткий smoke; долгое обучение не запущено.')


In [ ]:
SMOKE = PILOT.run_stage('smoke', PATHS, torch.device('cpu'))
print(json.dumps(SMOKE, ensure_ascii=False, indent=2))
print('Smoke artifact:', PATHS.output_dir / 'smoke')
print('STOP: пришли вывод smoke. Ячейки подтверждения и pilot пока НЕ запускай.')


In [ ]:
APPROVED_PILOT = False
if not (PATHS.output_dir / 'smoke' / 'complete').is_file():
    raise RuntimeError('Сначала нужен завершённый smoke; pilot не подтверждён')
if DEVICE != 'cuda':
    raise RuntimeError('Для долгого pilot нужен GPU; текущая среда не GPU')
answer = input('После проверки smoke введи RUN_KNOWN_K_TEMPORAL_PILOT для подтверждения обучения: ').strip()
if answer != 'RUN_KNOWN_K_TEMPORAL_PILOT':
    raise RuntimeError('Неверное подтверждение; обучение не запущено')
APPROVED_PILOT = True
print('Подтверждение принято. Следующая ячейка запустит долгое обучение на GPU.')


In [ ]:
if globals().get('APPROVED_PILOT') is not True:
    raise RuntimeError('Нет явного подтверждения pilot; сначала проверь smoke и ячейку подтверждения')
APPROVED_PILOT = False
PILOT_REPORT = PILOT.run_stage('pilot', PATHS, torch.device('cuda'))
print('Quality gate:', json.dumps(PILOT_REPORT.get('quality_gate'), ensure_ascii=False, indent=2))
print('Pilot artifact:', PATHS.output_dir / 'pilot')
print('Report:', json.dumps(PILOT_REPORT, ensure_ascii=False, indent=2))
print('STOP: пришли отчёт и gate; повторные seed, holdout и UI пока не запускаем.')
